# OCCAM Python Interface (pyoccam) Demonstration Notebook

This notebook demonstrates the complete workflow for using OCCAM through Python:
1. Loading data (with automatic test data detection)
2. Running search algorithms
3. Selecting best models by different criteria
4. Generating detailed fit reports
5. Analyzing results

**Note:** Run cells in order for best results.

In [4]:
# OCCAM Python - Simplified Demo Notebook
import pyoccam
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# Configuration
DATA_FILE = "dementia05_search_fullup.csv"  # or your data file
SEARCH_TYPE = "loopless-up"
SEARCH_LEVELS = 7
SEARCH_WIDTH = 3

# Initialize OCCAM
manager = pyoccam.VBMManager()
manager.set_report_separator(pyoccam.SPACESEP)

# Load data
if manager.init_from_command_line(["occam", DATA_FILE]):
    print(f"✓ Loaded: {DATA_FILE}")
    print(f"  Sample size: {manager.get_sample_size()}")
    print(f"  Variables: {', '.join(manager.get_variable_list())}")
    print(f"  Test data: {'Yes' if manager.has_test_data() else 'No'}")
else:
    print("ERROR: Could not load data file")

AttributeError: module 'pyoccam' has no attribute 'VBMManager'

In [2]:
# Run search
print(f"Running {SEARCH_TYPE} search...")
search_report = manager.generate_search_report(
    search_type=SEARCH_TYPE,
    levels=SEARCH_LEVELS,
    width=SEARCH_WIDTH,
    include_test_data=manager.has_test_data()
)

# Display search results directly in notebook
print("\n" + "="*80)
print("SEARCH RESULTS")
print("="*80)
print(search_report)

Configuration loaded successfully
Data file: SY_sample_pts_to_occam3_shuffle_split42_hdr.txt
Search type: loopless-up
Model selection: bic


In [3]:
# Parse search results into DataFrame for visualization
import re

# Extract table from search report (after the header lines)
lines = search_report.split('\n')
table_start = None
for i, line in enumerate(lines):
    if 'ID' in line and 'MODEL' in line and 'Level' in line:
        table_start = i
        break

if table_start:
    # Parse the table data
    data_lines = []
    for line in lines[table_start+1:]:
        if line.strip() and not line.startswith('Best Model'):
            # Parse the line - adjust based on your actual format
            parts = line.split()
            if len(parts) >= 10 and parts[0].isdigit():
                data_lines.append(parts)

    # Create DataFrame
    columns = ['ID', 'Model', 'Level', 'H', 'ddf', 'dLR', 'Alpha',
               'Info%', 'dAIC', 'dBIC', 'IncAlpha', 'PctCorrect']
    df = pd.DataFrame(data_lines, columns=columns[:len(data_lines[0])])

    # Convert numeric columns
    for col in ['Level', 'H', 'ddf', 'dLR', 'Alpha', 'dAIC', 'dBIC']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Display top models
    print("\nTop 5 Models by dBIC:")
    print(df.nlargest(5, 'dBIC')[['Model', 'Level', 'dBIC', 'dAIC']].to_string(index=False))

Created output directory: notebook_output


In [4]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'{SEARCH_TYPE.upper()} Search Results', fontsize=16, fontweight='bold')

# Get kept models for plotting
kept_models = manager.get_kept_models()

if kept_models:
    # Extract data for plotting
    levels = [m.level for m in kept_models]
    dbics = [m.dbic for m in kept_models]
    daics = [m.daic for m in kept_models]
    infos = [m.information * 100 for m in kept_models]  # Convert to percentage
    alphas = [m.alpha for m in kept_models]
    names = [m.name for m in kept_models]

    # Plot 1: dBIC vs Information Trade-off
    ax1 = axes[0, 0]
    scatter = ax1.scatter(infos, dbics, c=levels, cmap='viridis',
                         s=100, alpha=0.7, edgecolors='black')
    ax1.set_xlabel('Information Captured (%)')
    ax1.set_ylabel('dBIC (higher is better)')
    ax1.set_title('Model Quality Trade-off')
    ax1.grid(True, alpha=0.3)

    # Highlight best models
    best_bic_idx = np.argmax(dbics)
    ax1.scatter(infos[best_bic_idx], dbics[best_bic_idx],
               color='red', s=200, marker='*', zorder=5)
    ax1.annotate(names[best_bic_idx],
                xy=(infos[best_bic_idx], dbics[best_bic_idx]),
                xytext=(5, 5), textcoords='offset points', fontsize=8)

    plt.colorbar(scatter, ax=ax1, label='Level')

    # Plot 2: Information by Level
    ax2 = axes[0, 1]
    for level in set(levels):
        level_infos = [info for l, info in zip(levels, infos) if l == level]
        ax2.scatter([level] * len(level_infos), level_infos, alpha=0.6, s=80)
    ax2.set_xlabel('Search Level')
    ax2.set_ylabel('Information (%)')
    ax2.set_title('Information Captured by Level')
    ax2.grid(True, alpha=0.3)

    # Plot 3: Alpha Values (log scale)
    ax3 = axes[1, 0]
    # Filter out zero alphas for log scale
    non_zero_alphas = [(l, a) for l, a in zip(levels, alphas) if a > 0]
    if non_zero_alphas:
        nz_levels, nz_alphas = zip(*non_zero_alphas)
        ax3.scatter(nz_levels, nz_alphas, alpha=0.6, s=80)
        ax3.set_yscale('log')
    ax3.axhline(y=0.05, color='red', linestyle='--', label='α=0.05')
    ax3.set_xlabel('Search Level')
    ax3.set_ylabel('Alpha (log scale)')
    ax3.set_title('Model Significance by Level')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    # Plot 4: Model Progression
    ax4 = axes[1, 1]
    # Sort by level then dBIC for path
    sorted_models = sorted(zip(levels, dbics, names), key=lambda x: (x[0], x[1]))
    path_levels = [x[0] for x in sorted_models]
    path_dbics = [x[1] for x in sorted_models]
    ax4.plot(path_levels, path_dbics, 'o-', alpha=0.5, linewidth=1)
    ax4.set_xlabel('Search Level')
    ax4.set_ylabel('dBIC')
    ax4.set_title('Search Path Quality')
    ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print best models
print("\n" + "="*60)
print("BEST MODELS BY CRITERION:")
print("="*60)
print(f"BIC:         {manager.get_best_model_by_bic()}")
print(f"AIC:         {manager.get_best_model_by_aic()}")
print(f"Information: {manager.get_best_model_by_information()}")
print(f"Info+Alpha:  {manager.get_best_model_by_info_alpha()}")

Loading: SY_sample_pts_to_occam3_shuffle_split42_hdr.txt
✓ Data loaded successfully


In [5]:
# Get best model and generate fit report
best_model = manager.get_best_model_by_bic()
if not best_model:
    best_model = manager.get_best_model_by_information()

print(f"Generating fit report for: {best_model}")
print("="*80)

# Generate fit report
fit_report = manager.generate_fit_report(
    model_name=best_model,
    target_state="0"  # Adjust based on your data
)

# Display fit report inline
print(fit_report)

# Extract and display key statistics
print("\n" + "="*60)
print("MODEL SUMMARY:")
print("="*60)
model_stats = manager.get_model_statistics(best_model)
print(f"Model:              {model_stats.name}")
print(f"Information:        {model_stats.information:.4f} ({model_stats.information*100:.2f}%)")
print(f"dBIC:               {model_stats.dbic:.4f}")
print(f"dAIC:               {model_stats.daic:.4f}")
print(f"Alpha:              {model_stats.alpha:.6f}")
print(f"% Correct (train):  {model_stats.pct_correct_data:.2f}%")
if manager.has_test_data():
    print(f"% Correct (test):   {model_stats.pct_correct_test:.2f}%")

Data Statistics:
Sample size: 1077
Variables: 21
H(data): 10.7334
Test data: Present


✓ Test data detected - will include test performance metrics

Variables (21):
   1. ASpect_reclass
   2. CLay_reclass
   3. CurVature_reclass
  ...
  20. DrainageCl
  21. LS


In [6]:
# Compare top models side by side
top_models = []
for criterion, getter in [
    ("BIC", manager.get_best_model_by_bic),
    ("AIC", manager.get_best_model_by_aic),
    ("Info", manager.get_best_model_by_information)
]:
    model_name = getter()
    if model_name:
        stats = manager.get_model_statistics(model_name)
        top_models.append({
            'Criterion': criterion,
            'Model': model_name,
            'Info%': f"{stats.information*100:.2f}",
            'dBIC': f"{stats.dbic:.2f}",
            'dAIC': f"{stats.daic:.2f}",
            'Alpha': f"{stats.alpha:.4f}",
            '%Correct': f"{stats.pct_correct_data:.2f}"
        })

comparison_df = pd.DataFrame(top_models)
print("\nMODEL COMPARISON TABLE:")
print("="*80)
print(comparison_df.to_string(index=False))

# Save results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
comparison_df.to_csv(f"model_comparison_{timestamp}.csv", index=False)
print(f"\nResults saved to: model_comparison_{timestamp}.csv")

Reference model: bottom (independence)
Output format: space
Fit report target state: 0

✓ Configuration complete
